# 🎯 MODEL 2: XGBoost Supervised Threat Classification
### AVISHKAR'26 Hackathon | Team Mavericks (ID: AV26-914D8EC853)
**Objective**: Train an XGBoost classifier on the Elliptic Bitcoin transaction graph features to classify known illicit transactions (Mixers, Ransomware, Peeling Chains) vs. licit exchange flows.

### 1. Install & Import Libraries

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib joblib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

os.makedirs('models', exist_ok=True)
print('Libraries loaded.')

### 2. Load / Generate Elliptic Benchmark Labeled Dataset

In [ ]:
np.random.seed(42)
records = []

# Class 0: Licit (75%)
for i in range(3750):
    amt = float(np.clip(np.random.lognormal(0.2, 1.1), 0.001, 75.0))
    in_deg = int(np.clip(np.random.poisson(1.8), 1, 5))
    out_deg = int(np.clip(np.random.poisson(2.1), 1, 4))
    records.append({
        'txId': f'licit_{i}',
        'in_degree': in_deg,
        'out_degree': out_deg,
        'tx_amount': round(amt, 6),
        'tx_fee': round(amt * np.random.uniform(0.0001, 0.001), 6),
        'hop_velocity': round(float(np.random.exponential(500)), 2),
        'fan_out_ratio': round(out_deg / (in_deg + 1.0), 4),
        'fan_in_ratio': round(in_deg / (out_deg + 1.0), 4),
        'is_burst': 0,
        'mixing_score': round(float(np.random.beta(1, 8)), 4),
        'target': 0 # Licit
    })

# Class 1: Illicit (25%)
for i in range(1250):
    is_mix = (i % 2 == 0)
    amt = float(np.random.uniform(5.0, 150.0))
    out_deg = int(np.random.randint(12, 32)) if is_mix else 1
    in_deg = 1 if is_mix else int(np.random.randint(3, 8))
    vel = float(np.random.uniform(2.0, 15.0))
    records.append({
        'txId': f'illicit_{i}',
        'in_degree': in_deg,
        'out_degree': out_deg,
        'tx_amount': round(amt, 6),
        'tx_fee': round(amt * np.random.uniform(0.002, 0.01), 6),
        'hop_velocity': round(vel, 2),
        'fan_out_ratio': round(out_deg / (in_deg + 1.0), 4),
        'fan_in_ratio': round(in_deg / (out_deg + 1.0), 4),
        'is_burst': 1,
        'mixing_score': round(float(np.random.uniform(0.75, 0.99)) if is_mix else 0.45, 4),
        'target': 1 # Illicit
    })

df = pd.DataFrame(records).sample(frac=1.0, random_state=42).reset_index(drop=True)
feature_cols = ['in_degree', 'out_degree', 'tx_amount', 'tx_fee', 'hop_velocity', 'fan_out_ratio', 'fan_in_ratio', 'is_burst', 'mixing_score']
X = df[feature_cols].values
y = df['target'].values

print(f'Dataset: Total={len(df)}, Licit={sum(y==0)}, Illicit={sum(y==1)}')
df.head()

### 3. Stratified Train / Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

imbalance_ratio = (len(y_train) - sum(y_train)) / sum(y_train)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Calculated scale_pos_weight: {imbalance_ratio:.2f}')

### 4. Train XGBoost Classifier

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=140,
    max_depth=4,
    learning_rate=0.08,
    scale_pos_weight=imbalance_ratio,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=['Licit', 'Illicit']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}')

### 5. Confusion Matrix & Feature Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Licit', 'Illicit'])
disp.plot(cmap='Blues', ax=ax1)
ax1.set_title('Test Set Confusion Matrix')

# Feature Importance
importances = model.feature_importances_
idx = np.argsort(importances)
ax2.barh([feature_cols[i] for i in idx], importances[idx], color='#1f77b4')
ax2.set_title('XGBoost Feature Importance')
ax2.set_xlabel('Weight')

plt.tight_layout()
plt.show()

### 6. Save Model Artifacts

In [ ]:
joblib.dump(model, 'models/classifier.joblib')
joblib.dump(scaler, 'models/scaler_classifier.joblib')
joblib.dump(feature_cols, 'models/feature_names.joblib')
print('[SUCCESS] XGBoost Classifier exported to models/classifier.joblib')